In [1]:
import jax 
import jax.numpy as jnp

from probjax.core.custom_primitives.custom_inverse import custom_inverse
from functools import partial

@partial(custom_inverse, static_argnums=(1,))
def f(x, y):
    return x**y

f.definv(lambda y, x: y**(1./x))

x = jnp.array([1., 2., 3.])
y = f(x, 2.)

No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


In [2]:
jaxpr = jax.make_jaxpr(lambda x: f(x, 2.))(x)

In [3]:
jaxpr_forward = jaxpr.eqns[0].params["forward_jaxpr"]

jax.core.eval_jaxpr(jaxpr_forward.jaxpr, jaxpr_forward.literals, x)

[Array([1., 4., 9.], dtype=float32)]

In [4]:
inverse_forward = jaxpr.eqns[0].params["inverse_jaxpr"]

jax.core.eval_jaxpr(inverse_forward.jaxpr, inverse_forward.literals, y)

[Array([1., 2., 3.], dtype=float32), nan]

In [5]:
from probjax.utils.odeint import odeint, _odeint

drift = lambda t, x: -0.1*x
ys = odeint(drift, 1., jnp.linspace(0., 1., 100))

odeint.inv(drift, ys, jnp.linspace(0., 1., 100))
odeint.inv_and_logdet(drift, ys, jnp.linspace(0., 1., 100))

(100,)


(Array(1., dtype=float32), Array(0.1, dtype=float32))

In [9]:
from probjax.utils.odeint import odeint, _odeint
from probjax.core import inverse, inverse_and_logabsdet

ts = jnp.linspace(0., 1., 100)

def f(x):
    ys = odeint(lambda x, t: jnp.sin(x), x, ts, method="rk4")
    return ys[-1]


jaxpr = jax.make_jaxpr(f)(1.)

forward_jaxpr = jaxpr.eqns[0].params["forward_jaxpr"]
inverse_jaxpr = jaxpr.eqns[0].params["inverse_jaxpr"]

ys = f(1.)

In [10]:
inverse_and_logabsdet(f)(ys)

(Array(0.996, dtype=float32), Array(0., dtype=float32))

In [ ]:
jax.lax.dynamic_update_slice

Signature:
jax.lax.dynamic_update_slice(
    operand: Union[jax.Array, numpy.ndarray],
    update: Union[jax.Array, numpy.ndarray, numpy.bool_, numpy.number, bool, int, float, complex],
    start_indices: Union[jax.Array, collections.abc.Sequence[Union[jax.Array, numpy.ndarray, numpy.bool_, numpy.number, bool, int, float, complex]]],
) -> jax.Array
Docstring:
Wraps XLA's `DynamicUpdateSlice
<https://www.tensorflow.org/xla/operation_semantics#dynamicupdateslice>`_
operator.

Args:
  operand: an array to slice.
  update: an array containing the new values to write onto `operand`.
  start_indices: a list of scalar indices, one per dimension.

Returns:
  An array containing the slice.

Examples:
  Here is an example of updating a one-dimensional slice update:

  >>> x = jnp.zeros(6)
  >>> y = jnp.ones(3)
  >>> dynamic_update_slice(x, y, (2,))
  Array([0., 0., 1., 1., 1., 0.], dtype=float32)

  If the update slice is too large to fit in the array, the start
  index will be adjusted to make 

In [ ]:
def g(x):
    return x[1:5]

jaxpr = jax.make_jaxpr(g)(jnp.ones((10,)))

jaxpr.eqns[0]



a:f32[4] = dynamic_slice[slice_sizes=(4,)] b 1

In [ ]:
ys = jax.core.eval_jaxpr(forward_jaxpr.jaxpr, forward_jaxpr.literals, 1.)
ys

[Array([1.   , 1.   , 1.   , 1.   , 1.001, 1.001, 1.002, 1.002, 1.003,
        1.004, 1.005, 1.006, 1.007, 1.008, 1.01 , 1.011, 1.013, 1.014,
        1.016, 1.018, 1.02 , 1.022, 1.024, 1.026, 1.029, 1.031, 1.034,
        1.036, 1.039, 1.042, 1.045, 1.048, 1.051, 1.054, 1.058, 1.061,
        1.064, 1.068, 1.072, 1.076, 1.08 , 1.084, 1.088, 1.092, 1.096,
        1.1  , 1.105, 1.109, 1.114, 1.119, 1.124, 1.129, 1.134, 1.139,
        1.144, 1.149, 1.154, 1.16 , 1.165, 1.171, 1.177, 1.182, 1.188,
        1.194, 1.2  , 1.206, 1.213, 1.219, 1.225, 1.232, 1.238, 1.245,
        1.251, 1.258, 1.265, 1.272, 1.279, 1.286, 1.293, 1.3  , 1.307,
        1.315, 1.322, 1.329, 1.337, 1.345, 1.352, 1.36 , 1.368, 1.376,
        1.384, 1.392, 1.4  , 1.408, 1.416, 1.424, 1.432, 1.441, 1.449,
        1.458], dtype=float32)]

In [ ]:
inverse_jaxpr.literals

[Array([0.   , 0.01 , 0.02 , 0.03 , 0.04 , 0.051, 0.061, 0.071, 0.081,
        0.091, 0.101, 0.111, 0.121, 0.131, 0.141, 0.152, 0.162, 0.172,
        0.182, 0.192, 0.202, 0.212, 0.222, 0.232, 0.242, 0.253, 0.263,
        0.273, 0.283, 0.293, 0.303, 0.313, 0.323, 0.333, 0.343, 0.354,
        0.364, 0.374, 0.384, 0.394, 0.404, 0.414, 0.424, 0.434, 0.444,
        0.455, 0.465, 0.475, 0.485, 0.495, 0.505, 0.515, 0.525, 0.535,
        0.545, 0.556, 0.566, 0.576, 0.586, 0.596, 0.606, 0.616, 0.626,
        0.636, 0.646, 0.657, 0.667, 0.677, 0.687, 0.697, 0.707, 0.717,
        0.727, 0.737, 0.747, 0.758, 0.768, 0.778, 0.788, 0.798, 0.808,
        0.818, 0.828, 0.838, 0.848, 0.859, 0.869, 0.879, 0.889, 0.899,
        0.909, 0.919, 0.929, 0.939, 0.949, 0.96 , 0.97 , 0.98 , 0.99 ,
        1.   ], dtype=float32),
 Array([[0. , 0. , 0. , 0. ],
        [0.5, 0. , 0. , 0. ],
        [0. , 0.5, 0. , 0. ],
        [0. , 0. , 1. , 0. ]], dtype=float32),
 Array([0.167, 0.333, 0.333, 0.167], dtype=float32

In [ ]:
jax.core.eval_jaxpr(inverse_jaxpr.jaxpr, inverse_jaxpr.literals,ys[0])

[Array(0.996, dtype=float32), Array(0., dtype=float32)]